<div align="center">

# Trivia Night with KONASH

**Train a 4B model to answer multi-constraint trivia by searching Wikipedia**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/konaequity/openkona/blob/main/notebooks/trivia_night.ipynb)

</div>

---

Run all cells via **Runtime → Run all**. Requires a Colab GPU runtime — go to **Runtime → Change runtime type → T4 GPU** before running.

This notebook trains a **Qwen 3 4B** model to answer multi-constraint trivia questions. The agent learns to search a Wikipedia corpus, retrieve relevant evidence, and synthesize accurate answers. Training takes ~30 minutes on a T4.

---

In [ ]:
!pip install -qU "konash[train] @ git+https://github.com/konaequity/openkona.git"

In [ ]:
import os
import json
import urllib.request
import urllib.parse

import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("WARNING: No GPU detected. Training will be very slow.")
    print("Go to Runtime > Change runtime type > Select GPU.")

## Agentic Environment

The environment defines the knowledge the agent can search over. We fetch 8 Wikipedia articles covering the topics our trivia questions target and write them to disk as the agent's searchable corpus.

In [ ]:
def fetch_wikipedia_article(title: str) -> str:
    """Fetch plain-text extract of a Wikipedia article via the API."""
    params = urllib.parse.urlencode({
        "action": "query",
        "titles": title,
        "prop": "extracts",
        "explaintext": "1",
        "exsectionformat": "plain",
        "format": "json",
    })
    url = f"https://en.wikipedia.org/w/api.php?{params}"
    req = urllib.request.Request(url, headers={"User-Agent": "KONASH-TriviaNotebook/1.0"})
    with urllib.request.urlopen(req, timeout=30) as resp:
        data = json.loads(resp.read())
    pages = data.get("query", {}).get("pages", {})
    for page in pages.values():
        return page.get("extract", "")
    return ""


TRIVIA_TOPICS = [
    "Ancient Rome",
    "Rosetta Stone",
    "Periodic table",
    "DNA",
    "Mount Everest",
    "Ring of Fire",
    "Leonardo da Vinci",
    "Chess",
]

CORPUS_DIR = "./trivia_corpus"
os.makedirs(CORPUS_DIR, exist_ok=True)

print(f"Fetching {len(TRIVIA_TOPICS)} Wikipedia articles...")
for i, topic in enumerate(TRIVIA_TOPICS):
    text = fetch_wikipedia_article(topic)
    if text and len(text) > 200:
        words = text.split()
        if len(words) > 3000:
            text = " ".join(words[:3000])
        fname = topic.replace(" ", "_").replace("/", "_") + ".txt"
        with open(os.path.join(CORPUS_DIR, fname), "w") as f:
            f.write(text)
        print(f"  [{i+1}/{len(TRIVIA_TOPICS)}] {topic}: {len(text):,} chars")
    else:
        print(f"  [{i+1}/{len(TRIVIA_TOPICS)}] {topic}: SKIPPED (too short)")

## Create the Agent

We create a `konash.Agent` backed by Qwen 3 4B with 4-bit QLoRA quantization. This fits on a T4 GPU (16 GB VRAM). The agent will automatically chunk, embed, and index the corpus when training begins.

In [ ]:
import konash

agent = konash.Agent(
    base_model="Qwen/Qwen3-4B",
    corpus=CORPUS_DIR,
    project="trivia-night",
    chunk_size=384,
    load_in_4bit=True,
    gradient_checkpointing=True,
)

## Train

Each iteration, the agent synthesizes multi-constraint trivia questions from the corpus, generates search-and-answer rollouts, filters to questions at the learning frontier, and trains with OAPL. Two iterations is enough to see clear improvement.

In [ ]:
results = agent.train(iterations=2, rollouts_per_example=4)

## Play Trivia

The trained agent can now answer trivia questions by searching the Wikipedia corpus. Let's try it on a few questions it has never seen.

In [ ]:
TRIVIA_QUESTIONS = [
    # Frontier models get these wrong — answers require deep retrieval
    "What organization produced the first English translation of the Rosetta Stone's Greek text, and in what year?",
    "Who was the first person to publish the electron orbital filling rule known as the Aufbau principle, and in what year was it published?",
    "In Ancient Rome, wood planks from which specific mountain range were discovered 1,700 km away in Central Rome, and in what date range were the trees felled?",
    # More retrieval-hard questions (uncomment to try):
    # "Who was the first person to identify Mount Everest as the world's highest peak, and from which city was he stationed when he made the calculation?",
    # "What is the estimated number of legal positions in chess, expressed with its specific coefficient and confidence interval?",
    # "In the 1993 eruption of Lascar volcano along the Ring of Fire, how far did pyroclastic flows travel from the summit, and how far away was ash fall recorded?",
    # "What is the exact length in centimeters of the total female diploid nuclear genome per human cell?",
    # "What specific mathematical proof did Leonardo da Vinci contribute to, involving the doubling of a square's area, and in which of his notebooks does it appear?",
]

for q in TRIVIA_QUESTIONS:
    answer = agent.solve(q)
    print(f"Q: {q}")
    print(f"A: {answer}\n")

---

Your trivia agent is trained and ready! Check out the [KONASH GitHub](https://github.com/konaequity/openkona) for more examples.

**Next steps:**
- Add more Wikipedia articles to the corpus for broader coverage
- Run a 3rd training iteration for further improvement
- Deploy with vLLM + LoRA hot-swapping for production serving